# BirdCLEF+ 2026 Perch v2 Submission

Load the trained Perch probe artifact, embed hidden soundscape windows with the CPU Perch SavedModel, apply calibration, and write `submission.csv`. This notebook is intentionally scoring-only.


## 1. Setup


In [ ]:
from pathlib import Path
import json
import os
import random
import re
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    """Submission configuration for Perch v2 scoring."""

    seed = 42
    competition_name = "birdclef-2026"
    sample_rate = 32000
    duration = 5.0
    soundscape_seconds = 60
    embedding_dim = 1536
    hidden_dim = 512
    dropout = 0.25
    probe_batch_size = 128
    submission_batch_files = 16
    use_soundscape_priors = False
    hour_prior_weight = 0.20
    site_prior_weight = 0.10
    cooccurrence_prior_weight = 0.05
    cooccurrence_min_prob = 0.20
    prior_clip = 1.5
    tensorflow_wheel_root = Path(
        "/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0"
    )
    data_root = Path("/kaggle/input/competitions/birdclef-2026")
    perch_model_dir = Path(
        "/kaggle/input/models/google/"
        "bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"
    )
    artifact_dir = Path(
        "/kaggle/input/models/tuannm3812/"
        "birdclef-perch-v2-artifacts/pytorch/default/4/perch_v2"
    )


def seed_everything(seed: int = 42) -> None:
    """Set deterministic seeds."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def validate_data_root(path: Path) -> Path:
    """Validate the fixed BirdCLEF+ 2026 Kaggle competition input."""
    if (path / "train.csv").exists() or (
        path / "sample_submission.csv"
    ).exists():
        return path
    raise FileNotFoundError(
        f"BirdCLEF 2026 competition data not found at {path}."
    )


def package_version(package_name: str) -> str | None:
    """Return an installed package version when available."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def version_tuple(value: str) -> tuple[int, ...]:
    """Convert a version string to comparable integer components."""
    core = value.split("+")[0].split("-")[0]
    parts = []
    for part in core.split("."):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def install_tensorflow_220() -> None:
    """Install TensorFlow 2.20 from the attached Kaggle wheel input."""
    tf_wheels = sorted(
        CFG.tensorflow_wheel_root.glob("**/tensorflow-2.20.0*.whl")
    )
    tb_wheels = sorted(
        CFG.tensorflow_wheel_root.glob("**/tensorboard-2.20.0*.whl")
    )
    if not tf_wheels:
        raise FileNotFoundError(
            f"TensorFlow 2.20 wheel not found under {CFG.tensorflow_wheel_root}."
        )
    targets = [str(path) for path in tb_wheels[:1] + tf_wheels[:1]]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", *targets],
        check=True,
    )


seed_everything(CFG.seed)
CFG.data_root = validate_data_root(CFG.data_root)
print(f"Data root: {CFG.data_root}")
print(f"Artifact directory: {CFG.artifact_dir}")

import librosa
import soundfile as sf
import torch
from torch import nn
from tqdm.auto import tqdm

installed_tf = package_version("tensorflow")
if installed_tf is None or version_tuple(installed_tf) < version_tuple(
    "2.20.0"
):
    print(f"Installing TensorFlow 2.20 over TensorFlow {installed_tf}")
    install_tensorflow_220()

import tensorflow as tf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"TensorFlow: {tf.__version__}")


## 2. Load Artifacts


In [ ]:
class PerchProbe(nn.Module):
    """Shallow classifier trained on frozen Perch embeddings."""

    def __init__(self, embedding_dim: int, num_classes: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def torch_load_checkpoint(path: Path) -> dict[str, object]:
    """Load a PyTorch checkpoint across Kaggle torch versions."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_artifact_labels(artifact_dir: Path) -> list[str]:
    """Load labels ordered by model output index."""
    idx_to_label_raw = json.loads(
        (artifact_dir / "labels.json").read_text(encoding="utf-8")
    )
    return [idx_to_label_raw[str(i)] for i in range(len(idx_to_label_raw))]


def load_json_if_exists(path: Path) -> dict[str, object] | None:
    """Load a JSON artifact when present."""
    return (
        json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    )


labels = load_artifact_labels(CFG.artifact_dir)
label_to_idx = {label: idx for idx, label in enumerate(labels)}
checkpoint = torch_load_checkpoint(CFG.artifact_dir / "best_perch_probe.pt")
model = PerchProbe(CFG.embedding_dim, len(labels)).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()
calibration = load_json_if_exists(CFG.artifact_dir / "calibration.json")
soundscape_priors = load_json_if_exists(
    CFG.artifact_dir / "soundscape_priors.json"
)
if soundscape_priors and soundscape_priors.get("labels") != labels:
    print(
        "Soundscape prior label order does not match model labels; priors disabled."
    )
    soundscape_priors = None

print(f"Loaded labels: {len(labels):,}")
print(f"Calibration loaded: {calibration is not None}")
print(f"Soundscape priors loaded: {soundscape_priors is not None}")


## 3. Load CPU Perch


In [ ]:
def validate_perch_model_dir(path: Path) -> Path:
    """Validate the fixed Google CPU Perch SavedModel input."""
    if (path / "saved_model.pb").exists():
        return path
    raise FileNotFoundError(
        f"CPU Perch SavedModel not found at {path}. "
        "Attach google/bird-vocalization-classifier tensorflow2/perch_v2_cpu/1."
    )


def explain_perch_runtime_error(error: Exception) -> None:
    """Raise a clearer message for the CUDA-only Perch model on CPU."""
    message = str(error)
    if "current platform CPU" in message and "CUDA" in message:
        raise RuntimeError(
            "The attached Perch SavedModel requires CUDA. Attach perch_v2_cpu/1 for CPU submission."
        ) from error
    if "XlaCallModuleOp" in message:
        raise RuntimeError(
            "The attached Perch SavedModel is incompatible with this TensorFlow/XLA runtime."
        ) from error
    raise error


perch_model_dir = validate_perch_model_dir(CFG.perch_model_dir)
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Perch model: {perch_model_dir}")
print(f"Inputs: {infer.structured_input_signature}")


## 4. Score Soundscapes


In [ ]:
def softmax_numpy(logits: np.ndarray) -> np.ndarray:
    """Compute a stable NumPy softmax."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def parse_soundscape_context(value: object) -> dict[str, object]:
    """Extract site and hour from a soundscape filename or row id."""
    stem = Path(str(value)).stem
    stem = re.sub(r"_\d+(?:\.0)?$", "", stem)
    parts = stem.split("_")
    site = next(
        (part for part in parts if re.fullmatch(r"S\d+", part)), "unknown"
    )
    time_part = next(
        (part for part in reversed(parts) if re.fullmatch(r"\d{6}", part)),
        None,
    )
    hour = int(time_part[:2]) if time_part else -1
    return {"site": site, "hour": hour}


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    """Extract Perch embeddings for a batch of waveforms."""
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    try:
        if keyword_specs:
            input_name = next(iter(keyword_specs))
            outputs = infer(**{input_name: tensor})
        else:
            outputs = infer(tensor)
    except Exception as error:
        explain_perch_runtime_error(error)
    arrays = {name: np.asarray(value) for name, value in outputs.items()}
    embedding_name = next(
        (
            name
            for name in arrays
            if "embedding" in name.lower() or "embed" in name.lower()
        ),
        next(iter(arrays)),
    )
    value = arrays[embedding_name]
    if value.ndim == 3:
        value = value.mean(axis=1)
    elif value.ndim > 3:
        value = value.reshape(value.shape[0], -1)
    return value.astype(np.float32)


def build_audio_index() -> dict[str, Path]:
    """Index hidden-test audio files by filename stem."""
    audio_index = {}
    for folder in [
        CFG.data_root / "test_soundscapes",
        CFG.data_root / "test_audio",
    ]:
        if folder.exists():
            for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                for path in folder.glob(ext):
                    audio_index[path.stem] = path
    return audio_index


def read_soundscape(path: Path) -> np.ndarray:
    """Read a full soundscape file and normalize its length."""
    target_len = int(CFG.sample_rate * CFG.soundscape_seconds)
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != CFG.sample_rate:
        y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def infer_soundscape_files(paths: list[Path]) -> tuple[list[str], np.ndarray]:
    """Extract embeddings for all 5-second windows in soundscape files."""
    row_ids = []
    chunks = []
    windows_per_file = int(CFG.soundscape_seconds / CFG.duration)
    batch_files = int(CFG.submission_batch_files)
    iterator = range(0, len(paths), batch_files)
    iterator = tqdm(
        iterator,
        total=(len(paths) + batch_files - 1) // batch_files,
        desc="perch files",
    )
    for start in iterator:
        batch_paths = paths[start : start + batch_files]
        waveforms = np.empty(
            (
                len(batch_paths) * windows_per_file,
                int(CFG.sample_rate * CFG.duration),
            ),
            dtype=np.float32,
        )
        cursor = 0
        for audio_path in batch_paths:
            y = read_soundscape(audio_path)
            windows = y.reshape(windows_per_file, -1)
            waveforms[cursor : cursor + windows_per_file] = windows
            row_ids.extend(
                f"{audio_path.stem}_{end_time}"
                for end_time in range(5, CFG.soundscape_seconds + 1, 5)
            )
            cursor += windows_per_file
        chunks.append(run_perch_batch(waveforms))
    return row_ids, np.concatenate(chunks, axis=0)


def prior_offsets_for_rows(
    row_ids: list[str], probs: np.ndarray
) -> np.ndarray:
    """Build optional row-specific prior offsets for submission logits."""
    offsets = np.zeros((len(row_ids), len(labels)), dtype=np.float32)
    if not CFG.use_soundscape_priors or not soundscape_priors:
        return offsets
    hour_offsets = soundscape_priors.get("hour_offsets", {})
    site_offsets = soundscape_priors.get("site_offsets", {})
    co_counts = soundscape_priors.get("cooccurrence_counts", {})
    for row_idx, row_id in enumerate(row_ids):
        context = parse_soundscape_context(row_id)
        hour_key = str(context["hour"])
        site_key = str(context["site"])
        if hour_key in hour_offsets:
            offsets[row_idx] += CFG.hour_prior_weight * np.asarray(
                hour_offsets[hour_key], dtype=np.float32
            )
        if site_key in site_offsets:
            offsets[row_idx] += CFG.site_prior_weight * np.asarray(
                site_offsets[site_key], dtype=np.float32
            )
        active = np.flatnonzero(probs[row_idx] >= CFG.cooccurrence_min_prob)
        for active_idx in active:
            active_label = labels[int(active_idx)]
            for linked_label, count in co_counts.get(active_label, {}).items():
                if linked_label in label_to_idx:
                    boost = np.log1p(float(count)) / 10.0
                    offsets[row_idx, label_to_idx[linked_label]] += (
                        CFG.cooccurrence_prior_weight * boost
                    )
    return np.clip(offsets, -CFG.prior_clip, CFG.prior_clip)


@torch.no_grad()
def predict_embeddings(
    embeddings: np.ndarray, row_ids: list[str] | None = None
) -> np.ndarray:
    """Predict class probabilities for Perch embeddings."""
    logits_chunks = []
    for start in range(0, len(embeddings), CFG.probe_batch_size):
        batch = torch.from_numpy(
            embeddings[start : start + CFG.probe_batch_size]
        ).to(device)
        logits_chunks.append(model(batch).cpu().numpy())
    logits = np.concatenate(logits_chunks, axis=0)
    temperature = 1.0
    if calibration is not None:
        temperature = float(calibration.get("temperature", 1.0))
    logits = logits / max(temperature, 1e-6)
    probs = softmax_numpy(logits)
    if row_ids is not None and CFG.use_soundscape_priors and soundscape_priors:
        logits = logits + prior_offsets_for_rows(row_ids, probs)
    return softmax_numpy(logits)


sample_path = CFG.data_root / "sample_submission.csv"
sample_submission = pd.read_csv(sample_path)
target_columns = [col for col in sample_submission.columns if col != "row_id"]
target_positions = [
    idx for idx, label in enumerate(labels) if label in target_columns
]
target_labels = [labels[idx] for idx in target_positions]
audio_index = build_audio_index()

print(f"Sample submission: {sample_path}")
print(f"Rows: {len(sample_submission):,}")
print(f"Indexed audio files: {len(audio_index):,}")
print(f"Submission target columns: {len(target_columns):,}")
print(f"Model target overlap: {len(target_labels):,}")

submission = sample_submission.copy()
for col in target_columns:
    submission[col] = 0.0

if len(audio_index) == 0 and len(submission) <= 3:
    print(
        "Tiny public dry-run detected with no test audio; preserving sample submission values."
    )
    submission = sample_submission.copy()
else:
    test_paths = sorted(audio_index.values())
    row_ids, test_embeddings = infer_soundscape_files(test_paths)
    probs = predict_embeddings(test_embeddings, row_ids)[:, target_positions]
    pred_df = pd.DataFrame(probs, columns=target_labels)
    pred_df.insert(0, "row_id", row_ids)
    pred_df = pred_df.set_index("row_id")
    missing_rows = sorted(set(submission["row_id"]) - set(pred_df.index))
    if missing_rows:
        raise FileNotFoundError(
            f"Missing predictions for {len(missing_rows):,} submission rows. "
            f"First missing row_id={missing_rows[0]}"
        )
    submission[target_labels] = pred_df.loc[
        submission["row_id"], target_labels
    ].to_numpy()

submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Wrote submission: {submission_path}")
display(submission.head())
